# exp_10: Genus Selection Validation

**Purpose:** Validate genus sample counts after Phase 2c filtering and determine final genus list for Phase 3 experiments.

**Context:**
- Phase 1 filtered to 30 viable genera (≥500 samples per genus)
- Phase 2c applied proximity filter (5m) + outlier removal → ~20% sample reduction
- Risk: Some genera may have fallen below 500-sample threshold

**Approach:**
1. Load setup decisions from 03a (CHM, proximity, outlier, feature selection)
2. Apply all filtering strategies to get final experimental dataset
3. Validate sample counts (≥500 per genus in both cities)
4. Analyze genus separability in final 50-feature space
5. Group poorly separable genera (hierarchical clustering)
6. Export final genus configuration for Phase 3 experiments

**Output:**
- `genus_selection_final.json` - Final genus list with grouping decisions
- Visualizations: Sample counts, dendrogram, separability heatmap

**Runtime:** ~5-10 minutes (depends on clustering)

# ============================================================
# RUNTIME SETTINGS
# ============================================================
# Required: CPU (Standard)
# GPU: Not required
# High-RAM: Recommended (for hierarchical clustering)
#
# SETUP: Add GITHUB_TOKEN to Colab Secrets (key icon in sidebar)
# ============================================================

import subprocess
from google.colab import userdata

# Get GitHub token from Colab Secrets (for private repo access)
token = userdata.get("GITHUB_TOKEN")
if not token:
    raise ValueError(
        "GITHUB_TOKEN not found in Colab Secrets.\n"
        "1. Click the key icon in the left sidebar\n"
        "2. Add a secret named 'GITHUB_TOKEN' with your GitHub token\n"
        "3. Toggle 'Notebook access' ON"
    )

# Install package from private GitHub repo
repo_url = f"git+https://{token}@github.com/SilasPignotti/urban-tree-transfer.git"
subprocess.run(["pip", "install", repo_url, "-q"], check=True)

print("OK: Package installed")


In [ ]:
# Mount Google Drive for data files
from google.colab import drive

drive.mount("/content/drive")

print("Google Drive mounted")


In [ ]:
# Package imports
from urban_tree_transfer.config import MIN_SAMPLES_PER_GENUS, RANDOM_SEED
from urban_tree_transfer.experiments import ablation, data_loading
from urban_tree_transfer.utils import ExecutionLog, save_figure, setup_plotting

from pathlib import Path
from datetime import datetime, timezone
import json
import warnings
import gc

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist, squareform
from scipy.stats import entropy
from sklearn.preprocessing import StandardScaler

setup_plotting()
log = ExecutionLog("exp_10_genus_selection_validation")

warnings.filterwarnings('ignore', category=UserWarning)

print("OK: Package imports complete")


# ============================================================
# CONFIGURATION
# ============================================================

DRIVE_DIR = Path("/content/drive/MyDrive/dev/urban-tree-transfer")
INPUT_DIR = DRIVE_DIR / "data" / "phase_2_splits"
OUTPUT_DIR = DRIVE_DIR / "data" / "phase_3_experiments"

METADATA_DIR = OUTPUT_DIR / "metadata"
LOGS_DIR = OUTPUT_DIR / "logs"
FIGURES_DIR = OUTPUT_DIR / "figures" / "exp_10_genus_selection"

for d in [METADATA_DIR, LOGS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Analysis parameters
MIN_SAMPLES = MIN_SAMPLES_PER_GENUS  # 500
CLUSTERING_METHOD = 'ward'  # Ward linkage for hierarchical clustering
DISTANCE_THRESHOLD = 0.5  # Cut height for dendrogram (relative, 0-1 scale)
KL_THRESHOLD = 0.15  # Maximum KL-divergence after filtering

print(f"Input (Phase 2 Splits): {INPUT_DIR}")
print(f"Output (Phase 3):       {OUTPUT_DIR}")
print(f"Metadata:               {METADATA_DIR}")
print(f"Figures:                {FIGURES_DIR}")
print(f"Logs:                   {LOGS_DIR}")
print(f"Random seed:            {RANDOM_SEED}")
print(f"Min samples per genus:  {MIN_SAMPLES}")


In [ ]:
# File paths
SETUP_DECISIONS = BASE_PATH / "phase_3_experiments/metadata/setup_decisions.json"
BERLIN_TRAIN = BASE_PATH / "phase_2_splits/berlin_filtered_train.parquet"
LEIPZIG_FINETUNE = BASE_PATH / "phase_2_splits/leipzig_filtered_finetune.parquet"
OUTPUT_DIR = BASE_PATH / "phase_3_experiments/metadata"
FIGURE_DIR = BASE_PATH / "phase_3_experiments/figures/exp_10_genus_selection"

# Analysis parameters
MIN_SAMPLES = MIN_SAMPLES_PER_GENUS  # 500
CLUSTERING_METHOD = 'ward'  # Ward linkage for hierarchical clustering
DISTANCE_THRESHOLD = 0.5  # Cut height for dendrogram (relative, 0-1 scale)
KL_THRESHOLD = 0.15  # Maximum KL-divergence after filtering

# Create output directories
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print(f"✅ Configuration set")
print(f"📂 Output: {OUTPUT_DIR}")
print(f"📊 Figures: {FIGURE_DIR}")

# ============================================================
# SECTION 1: Load Setup Decisions & All Splits
# ============================================================

log.start_step("Load Setup & Data")

# Load setup decisions
setup_path = METADATA_DIR / "setup_decisions.json"

if not setup_path.exists():
    raise FileNotFoundError(
        f"setup_decisions.json not found at {setup_path}\n"
        "Run exp_09_feature_reduction.ipynb first to complete setup."
    )

with open(setup_path) as f:
    setup_config = json.load(f)

print("✅ Loaded setup decisions:")
print(f"   CHM Strategy: {setup_config['chm_strategy']['decision']}")
print(f"   Proximity Strategy: {setup_config['proximity_strategy']['decision']}")
print(f"   Outlier Strategy: {setup_config['outlier_strategy']['decision']}")
print(f"   Selected Features: {len(setup_config['selected_features'])}")

# Load ALL splits (not just train+finetune) for complete sample count validation
print("\nLoading ALL splits for sample count validation...")

# Use filtered splits (proximity applied)
variant = setup_config['proximity_strategy']['decision']

berlin_train, berlin_val, berlin_test = data_loading.load_berlin_splits(
    INPUT_DIR, variant=variant
)
leipzig_finetune, leipzig_test = data_loading.load_leipzig_splits(
    INPUT_DIR, variant=variant
)

print(f"\n✅ Data loaded ({variant} variant):")
print(f"   Berlin Train:      {len(berlin_train):,} samples")
print(f"   Berlin Val:        {len(berlin_val):,} samples")
print(f"   Berlin Test:       {len(berlin_test):,} samples")
print(f"   Leipzig Finetune:  {len(leipzig_finetune):,} samples")
print(f"   Leipzig Test:      {len(leipzig_test):,} samples")
print(f"   Total:             {len(berlin_train) + len(berlin_val) + len(berlin_test) + len(leipzig_finetune) + len(leipzig_test):,} samples")

log.end_step(status="success")


In [ ]:
# Load setup decisions from 03a
if not SETUP_DECISIONS.exists():
    raise FileNotFoundError(
        f"setup_decisions.json not found at {SETUP_DECISIONS}\n"
        "Please run 03a_setup_fixation.ipynb first!"
    )

with open(SETUP_DECISIONS) as f:
    setup_config = json.load(f)

print("✅ Loaded setup decisions:")
print(f"   CHM Strategy: {setup_config['chm_strategy']['decision']}")
print(f"   Proximity Strategy: {setup_config['proximity_strategy']['decision']}")
print(f"   Outlier Strategy: {setup_config['outlier_strategy']['decision']}")
print(f"   Feature Set: {setup_config['feature_set']['n_features']} features")
print(f"   Selected Features: {len(setup_config['selected_features'])}")

In [ ]:
# ============================================================
# SECTION 2: Apply Setup Decisions to All Splits
# ============================================================

log.start_step("Apply Setup Decisions")

print("Applying setup decisions to all splits...")

all_splits = [
    ("berlin_train", berlin_train),
    ("berlin_val", berlin_val),
    ("berlin_test", berlin_test),
    ("leipzig_finetune", leipzig_finetune),
    ("leipzig_test", leipzig_test),
]

processed_splits = {}

for split_name, split_df in all_splits:
    # Apply setup decisions
    city = "berlin" if "berlin" in split_name else "leipzig"
    split_type = split_name.split("_")[1]  # train/val/test/finetune
    
    df_processed = ablation.prepare_ablation_dataset(
        df=split_df,
        setup_decisions=setup_config,
        split_name=split_type
    )
    
    processed_splits[split_name] = df_processed
    print(f"   {split_name}: {len(split_df):,} → {len(df_processed):,} samples")

# Extract feature columns (exclude metadata)
metadata_cols = ['tree_id', 'city', 'genus_latin', 'species_latin', 'genus_german', 
                 'species_german', 'tree_type', 'plant_year', 'block_id',
                 'outlier_severity', 'position_corrected', 'correction_distance']
feature_cols = [col for col in processed_splits['berlin_train'].columns if col not in metadata_cols]

print(f"\n✅ Setup decisions applied")
print(f"   Feature columns: {len(feature_cols)}")
print(f"   Total processed samples: {sum(len(df) for df in processed_splits.values()):,}")

log.end_step(status="success")


## 4. Apply Setup Decisions (Final Dataset)

In [ ]:
# ============================================================
# SECTION 3: Sample Count Validation (All Splits)
# ============================================================

log.start_step("Sample Count Validation")

print("Validating sample counts across ALL splits...\n")

# Count samples per genus in each split
genus_counts = {}
for split_name, split_df in processed_splits.items():
    genus_counts[split_name] = split_df['genus_latin'].value_counts()

# Get all unique genera
all_genera = sorted(set().union(*[set(counts.index) for counts in genus_counts.values()]))

# Create comprehensive count table
count_df = pd.DataFrame({
    split_name: [genus_counts[split_name].get(g, 0) for g in all_genera]
    for split_name in processed_splits.keys()
}, index=all_genera)

# Add aggregate columns
count_df['Berlin_Total'] = count_df[['berlin_train', 'berlin_val', 'berlin_test']].sum(axis=1)
count_df['Leipzig_Total'] = count_df[['leipzig_finetune', 'leipzig_test']].sum(axis=1)
count_df['Min_Per_City'] = count_df[['Berlin_Total', 'Leipzig_Total']].min(axis=1)
count_df['Viable'] = count_df['Min_Per_City'] >= MIN_SAMPLES

print("Sample Counts per Genus (All Splits):")
print(count_df.to_string())

# Identify viable and excluded genera
viable_genera = count_df[count_df['Viable']].index.tolist()
excluded_genera = count_df[~count_df['Viable']].index.tolist()

print(f"\n✅ Sample count validation complete")
print(f"   Viable genera (≥{MIN_SAMPLES} in both cities): {len(viable_genera)}")
print(f"   Excluded genera (<{MIN_SAMPLES} in at least one city): {len(excluded_genera)}")
if excluded_genera:
    print(f"\n   Excluded genera:")
    for genus in excluded_genera:
        berlin_count = count_df.loc[genus, 'Berlin_Total']
        leipzig_count = count_df.loc[genus, 'Leipzig_Total']
        print(f"      {genus}: Berlin={berlin_count}, Leipzig={leipzig_count}")

log.end_step(status="success")


## 5. Sample Count Validation

In [ ]:
# Count samples per genus
berlin_counts = berlin_final['genus_latin'].value_counts().sort_index()
leipzig_counts = leipzig_final['genus_latin'].value_counts().sort_index()

# Get all genera present in data
all_genera = sorted(set(berlin_counts.index) | set(leipzig_counts.index))

# Create comparison table
count_df = pd.DataFrame({
    'Berlin': [berlin_counts.get(g, 0) for g in all_genera],
    'Leipzig': [leipzig_counts.get(g, 0) for g in all_genera],
}, index=all_genera)

count_df['Min'] = count_df.min(axis=1)
count_df['Viable'] = count_df['Min'] >= MIN_SAMPLES

print("Sample Counts per Genus:")
print(count_df.to_string())

# Identify viable and excluded genera
viable_genera = count_df[count_df['Viable']].index.tolist()
excluded_genera = count_df[~count_df['Viable']].index.tolist()

print(f"\n✅ Sample count validation complete")
print(f"   Viable genera (≥{MIN_SAMPLES} in both cities): {len(viable_genera)}")
print(f"   Excluded genera (<{MIN_SAMPLES} in at least one city): {len(excluded_genera)}")
if excluded_genera:
    print(f"   Excluded: {', '.join(excluded_genera)}")

In [ ]:
# Visualization: Sample count bar chart
fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(len(all_genera))
width = 0.35

berlin_vals = count_df['Berlin'].values
leipzig_vals = count_df['Leipzig'].values

ax.bar(x - width/2, berlin_vals, width, label='Berlin Train', alpha=0.8, color='#1f77b4')
ax.bar(x + width/2, leipzig_vals, width, label='Leipzig Finetune', alpha=0.8, color='#ff7f0e')
ax.axhline(y=MIN_SAMPLES, color='red', linestyle='--', linewidth=2, 
           label=f'Min Threshold ({MIN_SAMPLES})', zorder=3)

ax.set_xlabel('Genus', fontsize=12)
ax.set_ylabel('Sample Count', fontsize=12)
ax.set_title('Genus Sample Counts after Setup Decisions (Final Feature Set)', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(all_genera, rotation=45, ha='right')
ax.legend(loc='upper right')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
fig.savefig(FIGURE_DIR / "genus_sample_counts.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"✅ Figure saved: genus_sample_counts.png")

# ============================================================
# SECTION 4: Genus Separability Analysis (Train Data Only)
# ============================================================

log.start_step("Separability Analysis")

# Filter to viable genera only
berlin_train_viable = processed_splits['berlin_train'][processed_splits['berlin_train']['genus_latin'].isin(viable_genera)].copy()
leipzig_finetune_viable = processed_splits['leipzig_finetune'][processed_splits['leipzig_finetune']['genus_latin'].isin(viable_genera)].copy()

print(f"Filtered to viable genera:")
print(f"   Berlin Train:     {len(berlin_train_viable):,} samples ({len(viable_genera)} genera)")
print(f"   Leipzig Finetune: {len(leipzig_finetune_viable):,} samples ({len(viable_genera)} genera)")

# IMPORTANT: Only use train+finetune for separability analysis (not test data)
# Test data should not influence grouping decisions (no data leakage)
combined = pd.concat([berlin_train_viable, leipzig_finetune_viable], ignore_index=True)

print(f"\n✅ Combined dataset: {len(combined):,} samples")
print(f"   Note: Using only train+finetune splits for separability analysis")
print(f"   (Test data not used for grouping decisions to avoid data leakage)")

log.end_step(status="success")


In [ ]:
# Filter to viable genera only
berlin_viable = berlin_final[berlin_final['genus_latin'].isin(viable_genera)].copy()
leipzig_viable = leipzig_final[leipzig_final['genus_latin'].isin(viable_genera)].copy()

print(f"Filtered to viable genera:")
print(f"   Berlin: {len(berlin_viable):,} samples ({len(viable_genera)} genera)")
print(f"   Leipzig: {len(leipzig_viable):,} samples ({len(viable_genera)} genera)")

# Combine for genus centroid calculation
combined = pd.concat([berlin_viable, leipzig_viable], ignore_index=True)

print(f"\n✅ Combined dataset: {len(combined):,} samples")

In [ ]:
# Calculate genus centroids (mean feature values per genus)
print("Calculating genus centroids in final feature space...")

genus_centroids = combined.groupby('genus_latin')[feature_cols].mean()

print(f"✅ Genus centroids calculated")
print(f"   Shape: {genus_centroids.shape} ({len(viable_genera)} genera × {len(feature_cols)} features)")
print(f"\nFirst 3 genera centroids (first 5 features):")
print(genus_centroids.iloc[:3, :5].to_string())

In [ ]:
# Compute pairwise distances between genus centroids
print("Computing pairwise distances between genus centroids...")

# Standardize features (zero mean, unit variance) for fair distance calculation
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
centroids_scaled = scaler.fit_transform(genus_centroids)

# Compute pairwise Euclidean distances
distances_condensed = pdist(centroids_scaled, metric='euclidean')
distances_matrix = squareform(distances_condensed)

# Convert to DataFrame for easier handling
distances_df = pd.DataFrame(
    distances_matrix,
    index=genus_centroids.index,
    columns=genus_centroids.index
)

print(f"✅ Distance matrix computed")
print(f"   Shape: {distances_df.shape}")
print(f"   Distance range: [{distances_matrix[distances_matrix > 0].min():.2f}, {distances_matrix.max():.2f}]")
print(f"\nDistance matrix (first 5 genera):")
print(distances_df.iloc[:5, :5].round(2).to_string())

In [ ]:
# Visualization: Distance heatmap
fig, ax = plt.subplots(figsize=(12, 10))

sns.heatmap(
    distances_df,
    cmap='viridis',
    square=True,
    cbar_kws={'label': 'Euclidean Distance'},
    ax=ax
)

ax.set_title('Genus-Genus Separability (Euclidean Distance in Final Feature Space)', 
             fontsize=14, fontweight='bold', pad=20)
ax.set_xlabel('Genus', fontsize=12)
ax.set_ylabel('Genus', fontsize=12)

plt.tight_layout()
fig.savefig(FIGURE_DIR / "genus_separability_heatmap.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"✅ Figure saved: genus_separability_heatmap.png")

## 7. Hierarchical Clustering & Grouping

In [ ]:
# Perform hierarchical clustering
print(f"Performing hierarchical clustering (method: {CLUSTERING_METHOD})...")

linkage_matrix = linkage(distances_condensed, method=CLUSTERING_METHOD)

print(f"✅ Linkage matrix computed")
print(f"   Shape: {linkage_matrix.shape}")

In [ ]:
# Visualization: Dendrogram
fig, ax = plt.subplots(figsize=(14, 8))

# Normalize linkage distances to [0, 1] for interpretable threshold
max_distance = linkage_matrix[:, 2].max()
normalized_linkage = linkage_matrix.copy()
normalized_linkage[:, 2] = normalized_linkage[:, 2] / max_distance

dendrogram(
    normalized_linkage,
    labels=genus_centroids.index.tolist(),
    ax=ax,
    leaf_font_size=10
)

# Add threshold line
ax.axhline(
    y=DISTANCE_THRESHOLD, 
    color='red', 
    linestyle='--', 
    linewidth=2,
    label=f'Cut Threshold ({DISTANCE_THRESHOLD})'
)

ax.set_xlabel('Genus', fontsize=12)
ax.set_ylabel('Normalized Distance', fontsize=12)
ax.set_title('Genus Hierarchical Clustering (Ward Linkage)', fontsize=14, fontweight='bold')
ax.legend(loc='upper right')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
fig.savefig(FIGURE_DIR / "genus_dendrogram.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"✅ Figure saved: genus_dendrogram.png")

In [ ]:
# Identify clusters at threshold
from scipy.cluster.hierarchy import fcluster

print(f"Cutting dendrogram at threshold {DISTANCE_THRESHOLD}...")

cluster_labels = fcluster(normalized_linkage, t=DISTANCE_THRESHOLD, criterion='distance')

# Create genus -> cluster mapping
genus_to_cluster = dict(zip(genus_centroids.index, cluster_labels))

# Group genera by cluster
cluster_to_genera = {}
for genus, cluster_id in genus_to_cluster.items():
    if cluster_id not in cluster_to_genera:
        cluster_to_genera[cluster_id] = []
    cluster_to_genera[cluster_id].append(genus)

# Identify groups (clusters with >1 genus)
genus_groups = {}
for cluster_id, genera_list in cluster_to_genera.items():
    if len(genera_list) > 1:
        # Sort genera alphabetically
        genera_list_sorted = sorted(genera_list)
        group_name = f"GROUP_{cluster_id}"
        genus_groups[group_name] = genera_list_sorted

print(f"\n✅ Clustering complete")
print(f"   Total clusters: {len(cluster_to_genera)}")
print(f"   Singleton genera: {sum(1 for g in cluster_to_genera.values() if len(g) == 1)}")
print(f"   Multi-genus groups: {len(genus_groups)}")

if genus_groups:
    print(f"\nIdentified genus groups:")
    for group_name, genera_list in genus_groups.items():
        print(f"   {group_name}: {', '.join(genera_list)} ({len(genera_list)} genera)")
else:
    print(f"\n⚠️  No genus groups identified at threshold {DISTANCE_THRESHOLD}")
    print(f"   All genera are sufficiently distinct (no grouping needed)")

## 8. Create Final Genus List

In [ ]:
# Create final genus list (replace grouped genera with group names)
final_genera_list = []
genus_to_final = {}  # Mapping: original genus -> final name (genus or group)

# Add singleton genera
for cluster_id, genera_list in cluster_to_genera.items():
    if len(genera_list) == 1:
        genus = genera_list[0]
        final_genera_list.append(genus)
        genus_to_final[genus] = genus

# Add groups
for group_name, genera_list in genus_groups.items():
    final_genera_list.append(group_name)
    for genus in genera_list:
        genus_to_final[genus] = group_name

final_genera_list = sorted(final_genera_list)

print(f"✅ Final genus list created")
print(f"   Original viable genera: {len(viable_genera)}")
print(f"   Final classes (after grouping): {len(final_genera_list)}")
print(f"   Reduction: {len(viable_genera) - len(final_genera_list)} genera merged into groups")
print(f"\nFinal genus list:")
for i, name in enumerate(final_genera_list, 1):
    if name.startswith('GROUP_'):
        print(f"   {i:2d}. {name} ({len(genus_groups[name])} genera: {', '.join(genus_groups[name])})") 
    else:
        print(f"   {i:2d}. {name}")

In [ ]:
# Calculate sample counts for final classes
print("Calculating sample counts for final classes...")

# Map original genus to final class in dataframes
berlin_viable_mapped = berlin_viable.copy()
berlin_viable_mapped['final_class'] = berlin_viable_mapped['genus_latin'].map(genus_to_final)

leipzig_viable_mapped = leipzig_viable.copy()
leipzig_viable_mapped['final_class'] = leipzig_viable_mapped['genus_latin'].map(genus_to_final)

# Count samples per final class
berlin_final_counts = berlin_viable_mapped['final_class'].value_counts().sort_index()
leipzig_final_counts = leipzig_viable_mapped['final_class'].value_counts().sort_index()

final_counts_df = pd.DataFrame({
    'Berlin': berlin_final_counts,
    'Leipzig': leipzig_final_counts,
})
final_counts_df['Total'] = final_counts_df['Berlin'] + final_counts_df['Leipzig']

print(f"\n✅ Sample counts for final classes:")
print(final_counts_df.to_string())
print(f"\nTotal samples: {final_counts_df['Total'].sum():,}")

# ============================================================
# SECTION 6: Validate Split Stratification (KL-Divergence)
# ============================================================

log.start_step("KL-Divergence Validation")

print("Validating that genus filtering maintains split stratification...\n")

# Filter all splits to viable genera and apply genus mapping
splits_mapped = {}
for split_name, split_df in processed_splits.items():
    df = split_df[split_df['genus_latin'].isin(viable_genera)].copy()
    df['final_class'] = df['genus_latin'].map(genus_to_final)
    df = df[df['final_class'].notna()].copy()
    df['genus_latin'] = df['final_class']
    df.drop(columns=['final_class'], inplace=True)
    splits_mapped[split_name] = df
    print(f"   {split_name}: {len(split_df):,} → {len(df):,} samples")

# Calculate KL-divergence between splits
def compute_kl_divergence(split1_df, split2_df, class_col='genus_latin'):
    """Compute KL-divergence between class distributions."""
    dist1 = split1_df[class_col].value_counts(normalize=True).sort_index()
    dist2 = split2_df[class_col].value_counts(normalize=True).sort_index()
    
    # Align indices
    all_classes = sorted(set(dist1.index) | set(dist2.index))
    dist1 = dist1.reindex(all_classes, fill_value=1e-10)
    dist2 = dist2.reindex(all_classes, fill_value=1e-10)
    
    return entropy(dist1, dist2)

print(f"\nComputing KL-divergence between splits...")

# Berlin splits
kl_berlin_train_val = compute_kl_divergence(splits_mapped['berlin_train'], splits_mapped['berlin_val'])
kl_berlin_train_test = compute_kl_divergence(splits_mapped['berlin_train'], splits_mapped['berlin_test'])
kl_berlin_val_test = compute_kl_divergence(splits_mapped['berlin_val'], splits_mapped['berlin_test'])

# Leipzig splits
kl_leipzig_finetune_test = compute_kl_divergence(splits_mapped['leipzig_finetune'], splits_mapped['leipzig_test'])

kl_results = {
    "berlin": {
        "train_vs_val": kl_berlin_train_val,
        "train_vs_test": kl_berlin_train_test,
        "val_vs_test": kl_berlin_val_test
    },
    "leipzig": {
        "finetune_vs_test": kl_leipzig_finetune_test
    }
}

print(f"\nKL-Divergence Results (Threshold: {KL_THRESHOLD}):")
print(f"\nBerlin:")
print(f"   Train vs Val:  {kl_berlin_train_val:.4f} {'✅' if kl_berlin_train_val < KL_THRESHOLD else '❌'}")
print(f"   Train vs Test: {kl_berlin_train_test:.4f} {'✅' if kl_berlin_train_test < KL_THRESHOLD else '❌'}")
print(f"   Val vs Test:   {kl_berlin_val_test:.4f} {'✅' if kl_berlin_val_test < KL_THRESHOLD else '❌'}")

print(f"\nLeipzig:")
print(f"   Finetune vs Test: {kl_leipzig_finetune_test:.4f} {'✅' if kl_leipzig_finetune_test < KL_THRESHOLD else '❌'}")

# Check if all pass
all_kl_values = [
    kl_berlin_train_val, kl_berlin_train_test, kl_berlin_val_test,
    kl_leipzig_finetune_test
]
if all(kl < KL_THRESHOLD for kl in all_kl_values):
    print(f"\n✅ All KL-divergences below threshold - genus filtering maintains stratification!")
else:
    print(f"\n⚠️  Some KL-divergences exceed threshold - review genus filtering impact")

log.end_step(status="success")


In [ ]:
# Load all splits for KL-divergence validation
print("Loading all splits for KL-divergence validation...")

berlin_val = pd.read_parquet(BASE_PATH / "phase_2_splits/berlin_filtered_val.parquet")
berlin_test = pd.read_parquet(BASE_PATH / "phase_2_splits/berlin_filtered_test.parquet")
leipzig_test = pd.read_parquet(BASE_PATH / "phase_2_splits/leipzig_filtered_test.parquet")

# Apply setup decisions to all splits
berlin_val_final = prepare_ablation_dataset(berlin_val, setup_config, "val")
berlin_test_final = prepare_ablation_dataset(berlin_test, setup_config, "test")
leipzig_test_final = prepare_ablation_dataset(leipzig_test, setup_config, "test")

# Filter to viable genera and map to final classes
for df in [berlin_val_final, berlin_test_final, leipzig_test_final]:
    df = df[df['genus_latin'].isin(viable_genera)]
    df['final_class'] = df['genus_latin'].map(genus_to_final)

print(f"✅ All splits loaded and processed")

In [ ]:
# Calculate KL-divergence between splits
def compute_kl_divergence(split1_df, split2_df, class_col='final_class'):
    """Compute KL-divergence between class distributions of two splits."""
    # Get class distributions
    dist1 = split1_df[class_col].value_counts(normalize=True).sort_index()
    dist2 = split2_df[class_col].value_counts(normalize=True).sort_index()
    
    # Align indices (ensure same classes)
    all_classes = sorted(set(dist1.index) | set(dist2.index))
    dist1 = dist1.reindex(all_classes, fill_value=1e-10)
    dist2 = dist2.reindex(all_classes, fill_value=1e-10)
    
    # Compute KL-divergence
    kl = entropy(dist1, dist2)
    return kl

print("Computing KL-divergence between splits...\n")

# Berlin splits
kl_berlin_train_val = compute_kl_divergence(berlin_viable_mapped, berlin_val_final)
kl_berlin_train_test = compute_kl_divergence(berlin_viable_mapped, berlin_test_final)
kl_berlin_val_test = compute_kl_divergence(berlin_val_final, berlin_test_final)

# Leipzig splits
kl_leipzig_finetune_test = compute_kl_divergence(leipzig_viable_mapped, leipzig_test_final)

kl_results = {
    "berlin": {
        "train_vs_val": kl_berlin_train_val,
        "train_vs_test": kl_berlin_train_test,
        "val_vs_test": kl_berlin_val_test
    },
    "leipzig": {
        "finetune_vs_test": kl_leipzig_finetune_test
    }
}

print(f"KL-Divergence Results (Threshold: {KL_THRESHOLD}):")
print(f"\nBerlin:")
print(f"   Train vs Val:  {kl_berlin_train_val:.4f} {'✅' if kl_berlin_train_val < KL_THRESHOLD else '❌'}")
print(f"   Train vs Test: {kl_berlin_train_test:.4f} {'✅' if kl_berlin_train_test < KL_THRESHOLD else '❌'}")
print(f"   Val vs Test:   {kl_berlin_val_test:.4f} {'✅' if kl_berlin_val_test < KL_THRESHOLD else '❌'}")

print(f"\nLeipzig:")
print(f"   Finetune vs Test: {kl_leipzig_finetune_test:.4f} {'✅' if kl_leipzig_finetune_test < KL_THRESHOLD else '❌'}")

# Check if all pass
all_kl_values = [
    kl_berlin_train_val, kl_berlin_train_test, kl_berlin_val_test,
    kl_leipzig_finetune_test
]
if all(kl < KL_THRESHOLD for kl in all_kl_values):
    print(f"\n✅ All KL-divergences below threshold - genus filtering maintains stratification!")
else:
    print(f"\n⚠️  Some KL-divergences exceed threshold - review genus filtering impact")

## 10. Export Final Configuration

In [ ]:
# Build final configuration JSON
output_config = {
    "version": "1.0",
    "created": datetime.now(timezone.utc).isoformat(),
    "notebook": "exp_10_genus_selection_validation.ipynb",
    "config": {
        "min_samples_per_genus": MIN_SAMPLES,
        "clustering_method": CLUSTERING_METHOD,
        "distance_threshold": DISTANCE_THRESHOLD,
        "kl_threshold": KL_THRESHOLD,
        "filtering_stage": "post_setup_decisions"
    },
    "setup_decisions_applied": {
        "chm_strategy": setup_config['chm_strategy']['decision'],
        "proximity_strategy": setup_config['proximity_strategy']['decision'],
        "outlier_strategy": setup_config['outlier_strategy']['decision'],
        "n_features": setup_config['feature_set']['n_features']
    },
    "phase_1_genera": {
        "count": len(all_genera),
        "list": all_genera
    },
    "sample_count_analysis": {
        "berlin_train_total": len(berlin_final),
        "leipzig_finetune_total": len(leipzig_final),
        "genera_in_data": len(all_genera),
        "sample_counts": {
            "berlin": berlin_counts.to_dict(),
            "leipzig": leipzig_counts.to_dict()
        }
    },
    "validation_results": {
        "viable_genera": viable_genera,
        "excluded_genera": excluded_genera,
        "n_viable": len(viable_genera),
        "n_excluded": len(excluded_genera),
        "exclusion_reasons": {
            genus: f"Berlin: {berlin_counts.get(genus, 0)}, Leipzig: {leipzig_counts.get(genus, 0)}"
            for genus in excluded_genera
        } if excluded_genera else {}
    },
    "grouping_analysis": {
        "clustering_method": CLUSTERING_METHOD,
        "distance_threshold": DISTANCE_THRESHOLD,
        "n_clusters": len(cluster_to_genera),
        "n_groups": len(genus_groups),
        "genus_groups": genus_groups,
        "genus_to_final_mapping": genus_to_final
    },
    "decision": {
        "strategy_applied": "exclude_low_sample_and_group_similar",
        "final_genera_count": len(final_genera_list),
        "final_genera_list": final_genera_list,
        "grouping_applied": len(genus_groups) > 0,
        "reasoning": (
            f"{len(viable_genera)} genera have ≥{MIN_SAMPLES} samples in both cities. "
            f"Hierarchical clustering ({CLUSTERING_METHOD}) identified {len(genus_groups)} "
            f"groups of poorly separable genera (distance < {DISTANCE_THRESHOLD}). "
            f"Final dataset has {len(final_genera_list)} classes."
        )
    },
    "impact_assessment": {
        "berlin_samples_retained": len(berlin_viable),
        "leipzig_samples_retained": len(leipzig_viable),
        "retention_rate_berlin": len(berlin_viable) / len(berlin_final),
        "retention_rate_leipzig": len(leipzig_viable) / len(leipzig_final),
        "final_class_counts": {
            "berlin": berlin_final_counts.to_dict(),
            "leipzig": leipzig_final_counts.to_dict()
        }
    },
    "kl_divergence_validation": kl_results
}

# Export to JSON
output_file = OUTPUT_DIR / "genus_selection_final.json"
with open(output_file, 'w') as f:
    json.dump(output_config, f, indent=2)

print(f"✅ Configuration exported: {output_file}")
print(f"\nFinal Summary:")
print(f"   Original genera (Phase 1): {len(all_genera)}")
print(f"   Viable genera (≥{MIN_SAMPLES}): {len(viable_genera)}")
print(f"   Excluded genera: {len(excluded_genera)}")
print(f"   Genus groups formed: {len(genus_groups)}")
print(f"   Final classes: {len(final_genera_list)}")
print(f"   Total samples retained: {len(berlin_viable) + len(leipzig_viable):,}")
print(f"   Retention rate: {(len(berlin_viable) + len(leipzig_viable)) / (len(berlin_final) + len(leipzig_final)) * 100:.1f}%")

## 11. Summary & Next Steps

In [ ]:
print("="*80)
print("GENUS SELECTION VALIDATION - COMPLETE")
print("="*80)

print(f"\n📊 Results:")
print(f"   • Original genera (Phase 1): {len(all_genera)}")
print(f"   • Viable after Phase 2c: {len(viable_genera)}")
print(f"   • Final classes (after grouping): {len(final_genera_list)}")
print(f"   • Samples retained: {(len(berlin_viable) + len(leipzig_viable)) / (len(berlin_final) + len(leipzig_final)) * 100:.1f}%")

print(f"\n📁 Outputs:")
print(f"   • genus_selection_final.json")
print(f"   • genus_sample_counts.png")
print(f"   • genus_separability_heatmap.png")
print(f"   • genus_dendrogram.png")

print(f"\n🔄 Next Steps:")
print(f"   1. Review genus groups and validate biological plausibility")
print(f"   2. Run exp_11 (Algorithm Comparison) with final genus configuration")
print(f"   3. Update runner notebooks (03b, 03c, 03d) to load genus_selection_final.json")
print(f"   4. Document grouping decisions in methodological extensions")

print(f"\n✅ All validations passed - ready for Phase 3 experiments!")
print("="*80)